In [1]:
import torch
import os
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
from PIL import Image

In [2]:
# image load => transform => dataset of all images
class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations
        # all image path
        self.all_img_path  = [os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]
    def __len__(self):
        return len(self.all_img_path)
    def __getitem__(self,idx):
        img_path = self.all_img_path [idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)
        return img

In [3]:
root_dir_path = "./img_align_celeba"
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178 * 278 => 178*178
    transforms.Resize(64), # 64 * 64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # [-1,1]
])

In [4]:
dataset = ImageProcessor(root_dir_path,transformations)
dataloader = DataLoader(
    dataset,batch_size=128,shuffle=True
)

In [5]:
import numpy as np
import torch.nn as nn
import torch.optim as optim

In [6]:

class Generator(nn.Module):
    def __init__(self,z_dim =100, img_channel=3):
        super(Generator,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim,256), 
            nn.ReLU(),
            nn.Linear(256,512), 
            nn.ReLU(),
            nn.Linear(512,1024), 
            nn.ReLU(),
            nn.Linear(1024,64*64*img_channel), 
            nn.Tanh(), # [-1,1]  transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # [-1,1] taki yae match hoe
        )

    def forward(self,z):
        img= self.model(z)
        img = img.view(img.size(0),3,64,64)
        return img

In [7]:

class Discriminator(nn.Module):
    def __init__(self,z_dim =100, img_channel=3):
        super(Discriminator,self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(), # 4d tensor => 2d
            nn.Linear(img_channel*64*64,1024), 
            nn.LeakyReLU(0.2,inplace=True),
            nn.Linear(1024,512), 
            nn.LeakyReLU(0.2,inplace=True),
            nn.Linear(512,256), 
            nn.LeakyReLU(0.2,inplace=True),
            nn.Linear(256,1), 
            nn.Sigmoid(), # [-1,1]  
        )

    def forward(self,img):
        return  self.model(img)

In [8]:
GAN_loss = nn.BCELoss()
G = Generator()
g_optimizer = optim.Adam(G.parameters(),lr = 0.0002 , betas = (0.5 , 0.999))
D = Discriminator()
d_optimizer = optim.Adam(D.parameters(),lr = 0.0002 , betas = (0.5 , 0.999))

In [9]:
import torch
if torch.mps.is_available():
    device = torch.device("mps") 
elif torch.cuda.is_available:
    device = torch.device("cuda") 
else:
    device = torch.device("cpu") 
print(f"device is {device}")

device is mps


In [10]:
generator = G.to(device)
discriminator = D.to(device)

In [14]:
def train(generator,discriminator,dataloader,epochs=10):
    for epoch in range(epochs):
        for i,imgs in enumerate(dataloader):
            real_img = imgs.to(device)
            batch_size = real_img.size(0)
            # create real and fake labels.
            real_labels = torch.ones(batch_size,1).to(device)
            fake_labels = torch.zeros(batch_size,1).to(device)
            # train the discriminator
            d_optimizer.zero_grad()
            fake_imgs =  generator(torch.randn(batch_size,100).to(device))
            real_loss = GAN_loss(discriminator(real_img),(fake_labels))
            fake_loss = GAN_loss(discriminator(fake_imgs),(fake_labels))
            d_loss =  (real_loss + fake_loss)/2
            d_loss.backward()
            d_optimizer.step()

            # train the generator
            g_optimizer.zero_grad()
            fake_imgs =  generator(torch.randn(batch_size,100).to(device))
            g_loss =  GAN_loss(discriminator(fake_imgs),real_labels)
            g_loss.backward()
            g_optimizer.step()
            if i%50==0:
                print(f"for epochs {epoch+1}/{epochs} // batch : {i+1}.. ")
        # save genrated images for each epochs
        save_generated_images(generator,epoch,device)

In [15]:
import matplotlib.pyplot as plt
import torchvision
import numpy as np
def save_generated_images(generator,epoch,device,num_imgs=8):
    z = torch.randn(num_imgs,100).to(device)
    generated_img = generator(z).detach().cpu()
    # generated_img -> [-1,1] => [0,1] <- (normalize)
    grid = torchvision.utils.make_grid(generated_img, nrow=4,normalize=True)
    # why transpose ? => grid -> tensor => channels * height * width
    # but matplot => height * width * channels
    plt.imshow(np.transpose(grid,(1,2,0)))
    plt.title(f"epoch = {epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator,discriminator,dataloader,epochs=5)

for epochs 1/5 // batch : 1.. 
for epochs 1/5 // batch : 51.. 
for epochs 1/5 // batch : 101.. 
for epochs 1/5 // batch : 151.. 
for epochs 1/5 // batch : 201.. 
for epochs 1/5 // batch : 251.. 
for epochs 1/5 // batch : 301.. 
